In [1]:
import pandas as pd

notes_labeled = pd.read_parquet('notes_labeled_final.parquet')


print("First five lines")
print(notes_labeled.head())

print("\n all column：")
print(notes_labeled.columns.tolist())

First five lines
                noteId                                            summary  \
0  1783179305159200982  The House failed to pass a border protection l...   
1  1783182562279494134  TikTok only mentions “ban” and chooses to igno...   
2  1537142913737428992  Forbes has a good rundown of the investigation...   
3  1537145358521839617  They are expressing a personal opinion in a st...   
4  1537147343715282945  Teslas purchased after 12/31/19 are not eligib...   

  note_language                                      translated_en  \
0      eng_Latn  The House failed to pass a border protection l...   
1      eng_Latn  TikTok only mentions “ban” and chooses to igno...   
2      eng_Latn  Forbes has a good rundown of the investigation...   
3      eng_Latn  They are expressing a personal opinion in a st...   
4      eng_Latn  Teslas purchased after 12/31/19 are not eligib...   

                                        topic_labels  \
0                            [news_&_social

In [6]:
import pandas as pd
import os

print("Loading data...")
# 1. Load the labels (the parquet file) AND the raw notes (the 23-column TSV)
labels = pd.read_parquet('notes_labeled_final.parquet')  # Changed 'notes' to 'labels' here!
raw_notes = pd.read_csv('notes-00000.tsv', sep='\t')

# Load the rest of the raw data
ratings = pd.read_csv('ratings.tsv', sep='\t')
status = pd.read_csv('noteStatusHistory-00000.tsv', sep='\t')
enrollment = pd.read_csv('userEnrollment-00000.tsv', sep='\t')

cluster_col = 'policy_topic'
topics = labels[cluster_col].dropna().unique()

print(f"Found {len(topics)} unique topics. Starting to split...")

for topic in topics:
    safe_topic_name = str(topic).replace(" ", "_").replace("/", "_").replace(":", "")
    topic_dir = f"data_by_topic/{safe_topic_name}"
    os.makedirs(topic_dir, exist_ok=True)
    
    # Find the note IDs for this topic from the labeled parquet file
    valid_note_ids = labels[labels[cluster_col] == topic]['noteId'].unique()
    
    # FILTER THE RAW 23-COLUMN NOTES FILE, not the parquet file
    topic_notes = raw_notes[raw_notes['noteId'].isin(valid_note_ids)]
    
    topic_ratings = ratings[ratings['noteId'].isin(valid_note_ids)]
    topic_status = status[status['noteId'].isin(valid_note_ids)]
    
    valid_users = set(topic_ratings['raterParticipantId']).union(set(topic_status['noteAuthorParticipantId']))
    topic_enrollment = enrollment[enrollment['participantId'].isin(valid_users)]
    
    # Save the properly formatted files
    topic_notes.to_csv(f"{topic_dir}/notes.tsv", sep='\t', index=False)
    topic_ratings.to_csv(f"{topic_dir}/ratings.tsv", sep='\t', index=False)
    topic_status.to_csv(f"{topic_dir}/noteStatusHistory.tsv", sep='\t', index=False)
    topic_enrollment.to_csv(f"{topic_dir}/userEnrollment.tsv", sep='\t', index=False)
    
    print(f"Saved: {safe_topic_name} -> {len(topic_notes)} notes (23 columns)")

print("Step 1 Complete! All input files are ready.")

Loading data...


C:\Users\yuanxili\AppData\Local\Temp\ipykernel_24280\1320195089.py:11: DtypeWarning: Columns (0: lockedStatus, 1: currentMultiGroupStatus) have mixed types. Specify dtype option on import or set low_memory=False.
  status = pd.read_csv('noteStatusHistory-00000.tsv', sep='\t')


Found 53 unique topics. Starting to split...
Saved: 601_-_National_Way_of_Life_Positive -> 1630 notes (23 columns)
Saved: 303_-_Governmental_and_Administrative_Efficiency -> 1822 notes (23 columns)
Saved: 105_-_Military_Negative -> 8094 notes (23 columns)
Saved: 413_-_Nationalisation -> 7036 notes (23 columns)
Saved: 501_-_Environmental_Protection_Positive -> 722 notes (23 columns)
Saved: 401_-_Free_Market_Economy -> 2034 notes (23 columns)
Saved: 701_-_Labour_Groups_Positive -> 247 notes (23 columns)
Saved: 203_-_Constitutionalism_Positive -> 4242 notes (23 columns)
Saved: 107_-_Internationalism_Positive -> 574 notes (23 columns)
Saved: nan -> 9208 notes (23 columns)
Saved: 110_-_European_Community_Union_Negative -> 999 notes (23 columns)
Saved: 103_-_Anti-Imperialism -> 351 notes (23 columns)
Saved: 302_-_Centralisation -> 3753 notes (23 columns)
Saved: 102_-_Foreign_Special_Relationships_Negative -> 2329 notes (23 columns)
Saved: 606_-_Civic_Mindedness_Positive -> 1963 notes (23 col

In [7]:
import os
import subprocess
import time

# Configuration paths
data_root = "data_by_topic"
output_root = "output_by_topic"
python_executable = "py"

# Get all topic folders
topics = [f for f in os.listdir(data_root) if os.path.isdir(os.path.join(data_root, f))]
print(f"Found {len(topics)} topics. Preparing for batch processing...")

for topic in topics:
    data_dir = os.path.join(data_root, topic)
    out_dir = os.path.join(output_root, topic)
    
    # Core fix: Automatically create the output directory if it doesn't exist
    os.makedirs(out_dir, exist_ok=True) 
    
    files = {
        "--notes": os.path.join(data_dir, "notes.tsv"),
        "--ratings": os.path.join(data_dir, "ratings.tsv"),
        "--status": os.path.join(data_dir, "noteStatusHistory.tsv"),
        "--enrollment": os.path.join(data_dir, "userEnrollment.tsv")
    }

    # Check if all required input files exist
    if not all(os.path.exists(f) for f in files.values()):
        continue

    # Check ratings.tsv size again to ensure it contains data
    # Set to a 5KB threshold to ensure there are actual ratings recorded
    if os.path.getsize(files["--ratings"]) < 5000: 
        print(f"Skipping {topic}: Data volume too small.")
        continue

    print(f"\n>>> Processing topic: {topic}")
    start_time = time.time()

    # Build the execution command
    cmd = [python_executable, "main.py", "--outdir", out_dir]
    for flag, path in files.items():
        cmd.extend([flag, path])

    try:
        # Run the command and capture output/errors
        result = subprocess.run(cmd, capture_output=True, text=True)
        if result.returncode == 0:
            print(f"Success! Time elapsed: {time.time() - start_time:.2f} seconds")
        else:
            print(f"Topic {topic} execution failed, skipping.")
    except Exception as e:
        print(f"Execution error: {e}")

print("\nAll eligible topics have been processed! Please check the output_by_topic folder.")

Found 53 topics. Preparing for batch processing...

>>> Processing topic: 601_-_National_Way_of_Life_Positive
Success! Time elapsed: 37.62 seconds

>>> Processing topic: 303_-_Governmental_and_Administrative_Efficiency
Success! Time elapsed: 39.34 seconds

>>> Processing topic: 105_-_Military_Negative
Success! Time elapsed: 166.51 seconds

>>> Processing topic: 413_-_Nationalisation
Success! Time elapsed: 132.09 seconds

>>> Processing topic: 501_-_Environmental_Protection_Positive
Success! Time elapsed: 26.96 seconds

>>> Processing topic: 401_-_Free_Market_Economy
Success! Time elapsed: 41.85 seconds

>>> Processing topic: 701_-_Labour_Groups_Positive
Topic 701_-_Labour_Groups_Positive execution failed, skipping.

>>> Processing topic: 203_-_Constitutionalism_Positive
Success! Time elapsed: 86.83 seconds

>>> Processing topic: 107_-_Internationalism_Positive
Success! Time elapsed: 24.40 seconds

>>> Processing topic: nan
Success! Time elapsed: 164.41 seconds

>>> Processing topic: 11

KeyboardInterrupt: 